# ClinVar Strip Plot and ROC Curve

Generates two figures using variants that have ClinVar germline classifications:
- **Strip plot:** SGE fitness scores stratified by ClinVar classification (Benign → Pathogenic), with threshold lines overlaid.
- **ROC curve:** Classification performance of SGE score for B/LB vs. P/LP variants.

**Required data:** `scores` and `thresholds` sheets from the pipeline output Excel file (`*_data.xlsx`). The `scores` sheet must contain a `Germline classification` column (populated when the pipeline was run with a ClinVar input file).

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG/SVG require `vl-convert-python` (`pip install vl-convert-python`). Change the extension to `.html` for interactive output (no extra package needed).

In [ ]:
import pandas as pd
from sgeviz.figures import clinvar_strip

In [ ]:
# --- Configuration ---
excel_path = "GENE_data.xlsx"         # path to pipeline output Excel file
gene = "GENE"                         # gene name for plot title
strip_output = f"{gene}_clinvar_strip.png"  # change to .html for interactive output, or .svg
roc_output = f"{gene}_clinvar_roc.png"

# --- Plot customization (optional) ---
strip_width = 700     # strip plot width in pixels
strip_height = 250    # strip plot height in pixels
roc_width = 350       # ROC curve width in pixels
roc_height = 350      # ROC curve height in pixels

In [ ]:
# --- Load data ---
scores_df = pd.read_excel(excel_path, sheet_name="scores")
thresh_df = pd.read_excel(excel_path, sheet_name="thresholds")

thresholds = [
    thresh_df["non_functional_threshold"].iloc[0],
    thresh_df["functional_threshold"].iloc[0],
]

if "Germline classification" not in scores_df.columns:
    raise ValueError(
        "No 'Germline classification' column in scores sheet. "
        "Re-run the pipeline with a ClinVar input file to populate this column."
    )

clinvar_df = scores_df.dropna(subset=["Germline classification"]).copy()
print(f"Loaded {len(clinvar_df)} ClinVar-annotated variants. Thresholds: {thresholds}")

In [ ]:
# --- Strip plot ---
strip_chart = clinvar_strip.make_strip(clinvar_df, thresholds, gene=gene, width=strip_width, height=strip_height)
strip_chart

In [ ]:
# --- ROC curve ---
roc_chart = clinvar_strip.make_roc(clinvar_df, gene=gene, width=roc_width, height=roc_height)

if roc_chart is None:
    print("Insufficient B/LB and P/LP variants to compute ROC curve.")
else:
    roc_chart

In [ ]:
# --- Save ---
strip_chart.save(strip_output)
print(f"Saved: {strip_output}")

if roc_chart is not None:
    roc_chart.save(roc_output)
    print(f"Saved: {roc_output}")